In [0]:
from pyspark.sql.functions import col, lit, concat_ws, when

# ==========================================
# 03 - GENERIC DATA QUALITY VALIDATION
# ==========================================

dq_table = "mortgage_catalog.metadata.dq_rules"

# Get all tables having active DQ rules
tables = (
    spark.table(dq_table)
    .filter("is_active = true")
    .select("table_name")
    .distinct()
    .collect()
)

for row in tables:

    table_name = row["table_name"]

    source = f"mortgage_catalog.bronze.{table_name}"
    silver = f"mortgage_catalog.silver.{table_name.replace('_bronze', '')}_silver"
    quarantine = f"mortgage_catalog.bronze.{table_name.replace('_bronze', '')}_quarantine"

    print(f"\nProcessing DQ → {source}")

    # ----------------------------------
    # Check source
    # ----------------------------------
    if not spark.catalog.tableExists(source):
        print(f"SKIPPED → {source} not found")
        continue

    df = spark.table(source)

    # ----------------------------------
    # Get rules for current table
    # ----------------------------------
    rules = (
        spark.table(dq_table)
        .filter(
            (col("table_name") == table_name) &
            (col("is_active") == True)
        )
        .collect()
    )

    # Default status
    df = (
        df
        .withColumn("dq_status", lit("PASS"))
        .withColumn("dq_error", lit(None).cast("string"))
    )

    # ----------------------------------
    # Apply rules dynamically
    # ----------------------------------
    for rule in rules:

        column_name = rule["column_name"]
        rule_type = rule["rule_type"]
        rule_value = rule["rule_value"]
        error_message = rule["error_message"]

        # NOT NULL
        if rule_type == "NOT_NULL":

            fail_condition = col(column_name).isNull()

        # GREATER THAN
        elif rule_type == "GREATER_THAN":

            fail_condition = (
                col(column_name).cast("double")
                <= float(rule_value)
            )

        else:
            print(f"Unsupported rule → {rule_type}")
            continue

        # Set FAIL status
        df = df.withColumn(
            "dq_status",
            when(
                fail_condition,
                lit("FAIL")
            ).otherwise(col("dq_status"))
        )

        # Add error message
        df = df.withColumn(
            "dq_error",
            when(
                fail_condition,
                concat_ws(
                    " | ",
                    col("dq_error"),
                    lit(error_message)
                )
            ).otherwise(col("dq_error"))
        )

    # ----------------------------------
    # PASS → SILVER
    # ----------------------------------
    pass_df = df.filter(col("dq_status") == "PASS")

    (
        pass_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver)
    )

    # ----------------------------------
    # FAIL → QUARANTINE
    # ----------------------------------
    fail_df = df.filter(col("dq_status") == "FAIL")

    (
        fail_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(quarantine)
    )

    print(f"PASS       → {silver}")
    print(f"QUARANTINE → {quarantine}")

print("\nALL DQ VALIDATION COMPLETED ✅")


Processing DQ → mortgage_catalog.bronze.credit_score_bronze
PASS       → mortgage_catalog.silver.credit_score_silver
QUARANTINE → mortgage_catalog.bronze.credit_score_quarantine

Processing DQ → mortgage_catalog.bronze.loan_application_bronze
PASS       → mortgage_catalog.silver.loan_application_silver
QUARANTINE → mortgage_catalog.bronze.loan_application_quarantine

Processing DQ → mortgage_catalog.bronze.branch_bronze
PASS       → mortgage_catalog.silver.branch_silver
QUARANTINE → mortgage_catalog.bronze.branch_quarantine

Processing DQ → mortgage_catalog.bronze.document_bronze
PASS       → mortgage_catalog.silver.document_silver
QUARANTINE → mortgage_catalog.bronze.document_quarantine

Processing DQ → mortgage_catalog.bronze.property_bronze
PASS       → mortgage_catalog.silver.property_silver
QUARANTINE → mortgage_catalog.bronze.property_quarantine

Processing DQ → mortgage_catalog.bronze.payment_bronze
PASS       → mortgage_catalog.silver.payment_silver
QUARANTINE → mortgage_catal